> **AI disclosure:** This code was built by trial and error of multiple ai generated codes. The final code was debugged with the help of claude .The statistical results come from running this code on NASA POWER. Team members ran the notebook and exported the results.

# Kerala Trend Detective
**NASA Space Apps 2026 – Be An Earth System Trend Detective!**

A tool that lets you pick an Earth-science variable, a Kerala district and a season, then answers:
**what** is changing, **where**, **how much**, and whether the change is **statistically significant** (Mann-Kendall test, p < 0.05, with Sen's slope for the size of the change).

Data: NASA POWER (NASA Langley Research Center), monthly values built from NASA satellite and model products (MERRA-2 and GPM IMERG), free.

To use:Runtime → Run all. Then use the dropdowns in the last section.

In [ ]:
!pip -q install pymannkendall ipywidgets

In [ ]:
import requests, time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pymannkendall as mk

START, END = 1991, 2025          # analysis period (years)
ALPHA = 0.05                      # significance level

DISTRICTS = {  # approximate district centre (lat, lon), north to south
 "Kasaragod": (12.50, 74.99), "Kannur": (11.87, 75.37), "Wayanad": (11.69, 76.08),
 "Kozhikode": (11.26, 75.78), "Malappuram": (11.07, 76.07), "Palakkad": (10.78, 76.65),
 "Thrissur": (10.53, 76.21), "Ernakulam": (9.98, 76.28), "Idukki": (9.85, 76.97),
 "Kottayam": (9.59, 76.52), "Alappuzha": (9.49, 76.34), "Pathanamthitta": (9.26, 76.78),
 "Kollam": (8.89, 76.61), "Thiruvananthapuram": (8.52, 76.94)}

VARS = {  # label -> NASA POWER parameter code
 "Rainfall (mm/day)": "PRECTOTCORR",
 "Air temperature at 2 m (°C)": "T2M",
 "Root-zone soil wetness (0-1)": "GWETROOT",
 "Surface soil wetness (0-1)": "GWETTOP",
 "Relative humidity (%)": "RH2M"}

SEASONS = {
 "Annual": list(range(1, 13)),
 "Monsoon (Jun-Sep)": [6, 7, 8, 9],
 "Pre-monsoon (Mar-May)": [3, 4, 5],
 "Post-monsoon (Oct-Dec)": [10, 11, 12],
 "Winter (Jan-Feb)": [1, 2]}
print("Ready:", len(DISTRICTS), "districts,", len(VARS), "variables,", len(SEASONS), "seasons")

In [ ]:
URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
_cache = {}

def get_district(name):
    """Download monthly data for all variables for one district (cached)."""
    if name in _cache:
        return _cache[name]
    lat, lon = DISTRICTS[name]
    params = {"parameters": ",".join(VARS.values()), "community": "AG",
              "longitude": lon, "latitude": lat, "start": START, "end": END, "format": "JSON"}
    for attempt in range(3):
        try:
            r = requests.get(URL, params=params, timeout=90)
            r.raise_for_status()
            p = r.json()["properties"]["parameter"]
            break
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f"NASA POWER download failed for {name}: {e}")
            time.sleep(4)
    rows = []
    for code, series in p.items():
        for key, val in series.items():
            yr, mo = int(key[:4]), int(key[4:])
            if 1 <= mo <= 12 and val != -999:      # skip annual (13) and missing (-999)
                rows.append((code, yr, mo, val))
    df = pd.DataFrame(rows, columns=["var", "year", "month", "value"])
    _cache[name] = df
    return df

def seasonal_series(name, code, season):
    """One value per year: the mean over the months of the chosen season."""
    df = get_district(name)
    months = SEASONS[season]
    d = df[(df["var"] == code) & (df["month"].isin(months))]
    g = d.groupby("year")["value"].agg(["mean", "count"])
    return g[g["count"] == len(months)]["mean"]          # keep only complete seasons

def trend_stats(s):
    """Mann-Kendall significance + Sen's slope for a yearly series."""
    r = mk.original_test(s.values, alpha=ALPHA)
    n = len(s)
    total = r.slope * (n - 1)                              # change over the whole period
    pct = total / abs(s.mean()) * 100 if s.mean() != 0 else np.nan
    return {"n_years": n, "trend": r.trend, "significant": bool(r.h), "p_value": r.p,
            "slope_per_year": r.slope, "total_change": total, "pct_change": pct,
            "intercept": r.intercept, "mean": s.mean()}

def verdict(label, name, season, st, unit_hint=""):
    if st["significant"]:
        d = "increased" if st["trend"] == "increasing" else "decreased"
        return (f"{label} in {name} ({season}) has {d} significantly: about "
                f"{st['slope_per_year']:+.4f} per year, {st['total_change']:+.3f} over {st['n_years']} years "
                f"({st['pct_change']:+.1f}% of the mean). p = {st['p_value']:.3f}.")
    return (f"{label} in {name} ({season}) shows NO statistically significant trend "
            f"(p = {st['p_value']:.3f}); the apparent slope of {st['slope_per_year']:+.4f} per year "
            f"could be natural variability.")
print("Functions loaded")

In [ ]:
def plot_district(label, name, season):
    s = seasonal_series(name, VARS[label], season)
    st = trend_stats(s)
    x = np.arange(len(s))
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(s.index, s.values, "o-", color="#1f77b4", label="Observed (yearly)")
    ax.plot(s.index, st["intercept"] + st["slope_per_year"] * x, "--", lw=2.5,
            color="#d62728" if st["significant"] else "#7f7f7f",
            label=f"Sen's slope ({'significant' if st['significant'] else 'not significant'})")
    ax.set_title(f"{label} - {name} - {season}")
    ax.set_xlabel("Year"); ax.set_ylabel(label); ax.grid(alpha=.3); ax.legend()
    plt.tight_layout(); plt.show()
    print(verdict(label, name, season, st))
    return st

def compare_districts(label, season):
    rows = []
    for name in DISTRICTS:
        st = trend_stats(seasonal_series(name, VARS[label], season))
        rows.append({"district": name, **st})
    res = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ["#d62728" if (r.significant and r.slope_per_year > 0) else
              "#1f77b4" if r.significant else "#c7c7c7" for r in res.itertuples()]
    ax.barh(res["district"], res["slope_per_year"], color=colors)
    ax.invert_yaxis(); ax.axvline(0, color="k", lw=.8)
    ax.set_xlabel(f"Sen's slope of {label} per year")
    ax.set_title(f"{label} - {season}\nred = significant increase, blue = significant decrease, grey = not significant")
    plt.tight_layout(); plt.show()
    display(res[["district", "slope_per_year", "total_change", "pct_change", "p_value", "trend", "significant"]]
            .round(4).reset_index(drop=True))
    return res
print("Plot functions loaded")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

v = widgets.Dropdown(options=list(VARS), description="Variable:", layout=widgets.Layout(width="420px"))
s = widgets.Dropdown(options=list(SEASONS), description="Season:", layout=widgets.Layout(width="420px"))
d = widgets.Dropdown(options=list(DISTRICTS), description="District:", layout=widgets.Layout(width="420px"))
b1 = widgets.Button(description="Analyse this district", button_style="primary")
b2 = widgets.Button(description="Compare all districts", button_style="success")
out = widgets.Output()

def run_one(_):
    with out:
        clear_output(wait=True); print("Working..."); clear_output(wait=True)
        plot_district(v.value, d.value, s.value)
def run_all(_):
    with out:
        clear_output(wait=True); print("Downloading and analysing all districts...");
        clear_output(wait=True)
        compare_districts(v.value, s.value)
b1.on_click(run_one); b2.on_click(run_all)
display(widgets.VBox([v, s, d, widgets.HBox([b1, b2]), out]))

In [ ]:
rows = []
for name, (lat, lon) in DISTRICTS.items():
    for label, code_ in VARS.items():
        for season in SEASONS:
            ser = seasonal_series(name, code_, season)
            st = trend_stats(ser)
            rows.append({"district": name, "lat": lat, "lon": lon, "variable": label, "season": season,
                         **{k: (None if isinstance(x, float) and np.isnan(x) else x) for k, x in st.items()},
                         "years": [int(i) for i in ser.index], "values": [round(float(x), 4) for x in ser.values]})
res = pd.DataFrame(rows)
res.drop(columns=["years", "values"]).to_csv("kerala_trends.csv", index=False)
json.dump(json.loads(res.to_json(orient="records")), open("kerala_trends.json", "w"))
print(len(res), "results saved. Significant trends:", int(res["significant"].sum()))
try:
    from google.colab import files
    files.download("kerala_trends.json"); files.download("kerala_trends.csv")
except Exception:
    pass